# 🖥️ Bloque 6: HPC y Workflows en Linux

**Objetivo:** Entender cómo trabajar en entornos de computación de alto rendimiento (HPC) con Linux, SLURM y herramientas de paralelización.

---

## 1. Linux para Data Scientists — comandos esenciales

```bash
# Navegación
pwd                    # Directorio actual
ls -lah                # Listar con detalles y tamaños legibles
cd /ruta/al/proyecto   # Cambiar directorio
find . -name "*.py"    # Buscar archivos

# Gestión de procesos
top / htop             # Monitor de CPU y memoria en tiempo real
nvidia-smi             # Estado de las GPUs
ps aux | grep python   # Ver procesos Python activos
kill -9 <PID>          # Terminar un proceso

# Archivos y datos
wc -l datos.csv        # Contar líneas de un fichero
head -n 5 datos.csv    # Ver primeras 5 líneas
du -sh carpeta/        # Tamaño de una carpeta
rsync -avz src/ dest/  # Copiar archivos de forma eficiente

# Sesiones persistentes (muy importante en HPC)
tmux new -s mi_sesion  # Crear sesión que sobrevive al cierre de terminal
tmux attach -t mi_sesion  # Reconectarse
```

## 2. Gestión de entornos en Python

In [ ]:
# Este notebook cubre conceptos y comandos de shell
# Los bloques de código con '%%bash' se ejecutan en bash directamente en Jupyter

import subprocess
import sys
import os

print(f"Python: {sys.version}")
print(f"Sistema operativo: {os.name}")

### Entornos virtuales

```bash
# --- venv (built-in Python) ---
python -m venv mi_entorno
source mi_entorno/bin/activate     # Linux/Mac
mi_entorno\Scripts\activate        # Windows
pip install torch transformers
pip freeze > requirements.txt      # Guardar dependencias
deactivate

# --- conda (recomendado para HPC y GPU) ---
conda create -n ai-env python=3.10
conda activate ai-env
conda install pytorch torchvision pytorch-cuda=12.1 -c pytorch -c nvidia
conda env export > environment.yml  # Exportar entorno
conda env create -f environment.yml  # Recrear en otro servidor
```

## 3. SLURM — el gestor de colas en clústeres HPC

En un clúster HPC no ejecutas directamente tu script. Lo **encolas** con SLURM y el sistema lo lanza cuando hay recursos disponibles.

### Script SLURM típico para un job de entrenamiento:

```bash
#!/bin/bash
#SBATCH --job-name=train_model        # Nombre del job
#SBATCH --output=logs/job_%j.out      # Fichero de salida (%j = job ID)
#SBATCH --error=logs/job_%j.err       # Fichero de errores
#SBATCH --ntasks=1                    # Número de tareas
#SBATCH --cpus-per-task=8             # CPUs por tarea
#SBATCH --mem=32G                     # Memoria RAM
#SBATCH --gres=gpu:1                  # 1 GPU
#SBATCH --time=04:00:00               # Tiempo máximo (hh:mm:ss)
#SBATCH --partition=gpu               # Partición/cola

# Cargar módulos del entorno HPC
module load cuda/12.1
module load python/3.10

# Activar entorno conda
source activate ai-env

# Ejecutar script
python train.py --epochs 100 --batch-size 32 --lr 0.001
```

### Comandos SLURM más usados:

```bash
sbatch mi_job.sh           # Enviar job a la cola
squeue -u mi_usuario       # Ver estado de mis jobs
squeue -j <JOB_ID>         # Ver estado de un job específico
scancel <JOB_ID>           # Cancelar job
sacct -j <JOB_ID>          # Ver historial y estadísticas
sinfo                      # Ver particiones y nodos disponibles
```

## 4. Paralelización en Python

In [ ]:
import time
from multiprocessing import Pool
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from joblib import Parallel, delayed

def tarea_pesada(n):
    """Simula una tarea computacionalmente costosa"""
    return sum(i**2 for i in range(n))

tareas = [10_000] * 20  # 20 tareas

# --- Secuencial ---
t0 = time.time()
resultados_seq = [tarea_pesada(n) for n in tareas]
t_seq = time.time() - t0
print(f"Secuencial:           {t_seq:.3f}s")

# --- joblib (la más fácil de usar) ---
t0 = time.time()
resultados_par = Parallel(n_jobs=4)(delayed(tarea_pesada)(n) for n in tareas)
t_par = time.time() - t0
print(f"Paralelo (joblib x4): {t_par:.3f}s")
print(f"Speedup: {t_seq/t_par:.1f}x")

## 5. Monitorización de GPUs con Python

In [ ]:
import torch

if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f"GPUs disponibles: {n_gpus}")
    
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        mem_total = props.total_memory / 1e9
        mem_alloc = torch.cuda.memory_allocated(i) / 1e9
        print(f"\nGPU {i}: {props.name}")
        print(f"  Memoria total: {mem_total:.1f} GB")
        print(f"  Memoria usada: {mem_alloc:.2f} GB")
        print(f"  Compute Capability: {props.major}.{props.minor}")
else:
    print("No hay GPU disponible — usando CPU")
    print("\nEn producción/HPC, usar CUDA acelera entrenamiento 10-100x")

## 6. Buenas prácticas en HPC

```
✅ Siempre usa tmux o screen para sesiones largas
✅ Guarda checkpoints del modelo cada N epochs (el job puede cancelarse)
✅ Usa logging a fichero, no solo a stdout
✅ Estima recursos antes de pedir en SLURM (no pidas 1TB RAM para un modelo pequeño)
✅ Usa variables de entorno para rutas (no hardcodees /home/ruben/...)
✅ Versiona tus experimentos con MLflow o Weights & Biases
```

### Checkpoint de PyTorch:

```python
# Guardar checkpoint
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}, f'checkpoints/model_epoch_{epoch}.pt')

# Cargar checkpoint
checkpoint = torch.load('checkpoints/model_epoch_50.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1
```

---

## ✅ Resumen del bloque

- Conoces los **comandos Linux** esenciales para trabajar en servidores
- Sabes gestionar **entornos** con venv y conda
- Entiendes cómo funciona **SLURM** y cómo escribir un script de job
- Puedes **paralelizar** tareas en Python con joblib y multiprocessing
- Sabes monitorizar **GPUs** y guardar **checkpoints** del modelo

---

## ➡️ Siguiente paso

Continúa con el **Bloque 7: Despliegue en Producción** → `07_model_deployment.ipynb`